# 🎓 Advanced B.Tech Course-book RAG Assistant

Welcome to the **Advanced Retrieval-Augmented Generation (RAG)** pipeline notebook! This notebook is specifically engineered to handle complex, semi-structured documents like academic course books, credit schedules, and curriculum PDFs.

### 🚀 Advanced Techniques Implemented Here:
1. **Query Expansion (Pre-Retrieval)**: Uses the LLM to translate a user's short query into 4 distinct search queries (capturing acronyms, alternate wordings, etc.).
2. **Parent-Child Chunking**: We split the document into small child chunks (300 chars) for highly precise semantic matches, but retrieve the larger parent chunks (1200 chars) to provide complete context to the LLM.
3. **Hybrid Search (Dense + Sparse)**: Combines dense vector searches (Chroma DB) with sparse keyword searches (BM25 Retriever) to easily find exact course codes (e.g., CS-301) and semester names.
4. **Reciprocal Rank Fusion (RRF)**: Re-ranks and merges all search results using the state-of-the-art mathematical RRF algorithm.
5. **Citation Tracing**: The system tracks exact page numbers for every snippet used, strictly enforcing that the LLM cite its sources in inline markdown formats (e.g. `[Page 12]`).

Let's load our libraries and run this!

In [ ]:
import os
import sys
import json
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Check if at least one LLM API key is present
keys = ["GEMINI_API_KEY", "GOOGLE_API_KEY", "GROQ_API_KEY", "OPENAI_API_KEY"]
has_key = any(os.environ.get(k) for k in keys)
if not has_key:
    print("⚠️ WARNING: No cloud API keys found in your .env. The system will fall back to local Ollama.")
else:
    print("✅ Active API keys detected. Ready for RAG!")

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

## 🛠️ Step 1: Ingestion & Parent-Child Chunking
We load the PDF and chunk it recursively. Smaller chunks are sent to Chroma (Chroma persistent DB); larger parents are saved locally to `parent_store.json`.

In [ ]:
PDF_PATH = "/Users/vivekrajpurohit/practicemodel/ragmodel /Course-book_B.Tech_._summer2025-revised.pdf"
PERSIST_DIR = "/Users/vivekrajpurohit/practicemodel/ragmodel /chroma_db"
PARENT_STORE_PATH = "/Users/vivekrajpurohit/practicemodel/ragmodel /parent_store.json"

def load_and_index_pdf():
    # Load free local embeddings (runs completely offline on your Mac CPU)
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    
    # Check if index exists on disk
    if os.path.exists(PERSIST_DIR) and os.path.exists(PARENT_STORE_PATH):
        print("⚡ Loading existing persistent indices from disk...")
        vectorstore = Chroma(
            collection_name="btech_child_chunks",
            persist_directory=PERSIST_DIR,
            embedding_function=embeddings
        )
        with open(PARENT_STORE_PATH, "r") as f:
            parent_store = json.load(f)
        
        parent_docs = []
        for pid, data in parent_store.items():
            parent_docs.append(Document(page_content=data["content"], metadata=data["metadata"]))
        
        print(f"✅ Loaded {len(parent_store)} parent chunks successfully!")
        return vectorstore, parent_store, parent_docs

    print("🚀 Ingestion Pipeline Started: Parsing PDF...")
    loader = PyPDFLoader(PDF_PATH)
    pages = loader.load()
    print(f"📄 Loaded {len(pages)} pages from the PDF.")

    parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=200)
    child_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

    parent_store = {}
    parent_docs = []
    child_docs = []
    parent_idx = 0

    for page in pages:
        page_num = page.metadata.get("page", 0) + 1
        parents = parent_splitter.split_documents([page])
        for p in parents:
            parent_id = f"parent_{parent_idx}"
            parent_data = {
                "id": parent_id,
                "content": p.page_content,
                "metadata": {
                    "parent_id": parent_id,
                    "page": page_num,
                    "source": "Course-book_B.Tech"
                }
            }
            parent_store[parent_id] = parent_data
            
            p_doc = Document(page_content=p.page_content, metadata=parent_data["metadata"])
            parent_docs.append(p_doc)
            
            # Split parent chunk into child chunks
            children = child_splitter.split_text(p.page_content)
            for c_text in children:
                child_docs.append(Document(
                    page_content=c_text,
                    metadata={"parent_id": parent_id, "page": page_num}
                ))
            parent_idx += 1

    print(f"📦 Created {len(parent_docs)} parent chunks and {len(child_docs)} child chunks.")
    print("🧠 Indexing child chunks in Chroma Vector DB...")
    
    vectorstore = Chroma.from_documents(
        documents=child_docs,
        embedding=embeddings,
        collection_name="btech_child_chunks",
        persist_directory=PERSIST_DIR
    )
    
    with open(PARENT_STORE_PATH, "w") as f:
        json.dump(parent_store, f, indent=4)
        
    print("🎉 Ingestion complete!")
    return vectorstore, parent_store, parent_docs

vectorstore, parent_store, parent_docs = load_and_index_pdf()
# Initialize sparse BM25 retriever
bm25 = BM25Retriever.from_documents(parent_docs)

## 🤖 Step 2: Dynamic Multi-Provider LLM Selector
Selects and initializes your LLM based on whatever active keys exist in `.env` (Groq, Gemini, OpenAI, or local Ollama).

In [ ]:
def get_llm():
    # 1. Groq (Recommended Free Cloud Tier)
    if os.environ.get("GROQ_API_KEY"):
        from langchain_groq import ChatGroq
        print("🚀 Using Groq Cloud LLM (llama-3.1-8b-instant)...")
        return ChatGroq(model="llama-3.1-8b-instant", temperature=0.0)
        
    # 2. Google Gemini (Using your key)
    if os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY"):
        if os.environ.get("GEMINI_API_KEY") and not os.environ.get("GOOGLE_API_KEY"):
            os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]
        from langchain_google_genai import ChatGoogleGenerativeAI
        print("🚀 Using Google Gemini LLM (gemini-2.5-flash)...")
        return ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)
        
    # 3. OpenAI
    if os.environ.get("OPENAI_API_KEY") and not os.environ.get("OPENAI_API_KEY").startswith("sk-proj-expired"):
        from langchain_openai import ChatOpenAI
        print("🚀 Using OpenAI LLM (gpt-4o-mini)...")
        return ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

    # 4. Fallback to local Ollama
    from langchain_community.chat_models import ChatOllama
    print("🤖 No cloud keys found. Using local Ollama (llama3)...")
    return ChatOllama(model="llama3", temperature=0.0)

llm = get_llm()

## 🔄 Step 3: Query Expansion Layer
Generate search variations to increase query recall.

In [ ]:
def expand_query(query):
    prompt = f"""You are an advanced search system assistant.
Your task is to generate 3 alternative search queries based on the user's input query.
These queries should be optimized for a search engine retrieving academic syllabi, course codes, course structures, credits, and semesters from a B.Tech course book.
Do not write any introductory or concluding text, just list the 3 queries, one per line.

User Query: {query}

Alternative Queries:"""
    try:
        response = llm.invoke(prompt)
        queries = [q.strip().strip("-* ").strip() for q in response.content.split("\n") if q.strip()]
        queries.insert(0, query) # add original query at the beginning
        return list(dict.fromkeys([q for q in queries if q]))[:4]
    except Exception as e:
        print(f"⚠️ Query expansion failed: {e}")
        return [query]

test_query = "CS syllabus semester 3"
print("Search variations:", expand_query(test_query))

## 🔎 Step 4: Hybrid Search & Reciprocal Rank Fusion (RRF)
We merge dense and sparse matches. The dense search retrieves children and maps them to their parents; the sparse BM25 search queries parents directly. RRF scores the results.

In [ ]:
def reciprocal_rank_fusion(dense_results, sparse_results, k=60):
    rrf_scores = {}
    for rank, doc in enumerate(dense_results):
        parent_id = doc.metadata["parent_id"]
        if parent_id not in rrf_scores:
            rrf_scores[parent_id] = {"doc": doc, "score": 0.0}
        rrf_scores[parent_id]["score"] += 1.0 / (k + (rank + 1))
        
    for rank, doc in enumerate(sparse_results):
        parent_id = doc.metadata["parent_id"]
        if parent_id not in rrf_scores:
            rrf_scores[parent_id] = {"doc": doc, "score": 0.0}
        rrf_scores[parent_id]["score"] += 1.0 / (k + (rank + 1))
        
    sorted_items = sorted(rrf_scores.values(), key=lambda x: x["score"], reverse=True)
    return [item["doc"] for item in sorted_items]

def perform_hybrid_search(expanded_queries, top_n=4):
    dense_parents = []
    sparse_parents = []
    
    for q in expanded_queries:
        # Dense vector search
        child_matches = vectorstore.similarity_search(q, k=6)
        for child in child_matches:
            pid = child.metadata["parent_id"]
            parent_data = parent_store[pid]
            dense_parents.append(Document(page_content=parent_data["content"], metadata=parent_data["metadata"]))
            
        # Sparse keyword search
        sparse_matches = bm25.invoke(q)[:4]
        sparse_parents.extend(sparse_matches)
        
    return reciprocal_rank_fusion(dense_parents, sparse_parents)[:top_n]

## 🤖 Step 5: Grounded Answer Synthesis & Citations
We compile our context, inject it into an expert academic counselor prompt, and generate answers.

In [ ]:
def generate_answer(query, context_docs):
    context_text = ""
    for idx, doc in enumerate(context_docs):
        page = doc.metadata.get("page", "Unknown")
        context_text += f"--- [Document Chunk {idx+1} | Source Page: {page}] ---\n"
        context_text += f"{doc.page_content}\n\n"
        
    system_prompt = """You are an expert academic counselor and guide for the B.Tech program.
Your task is to answer the student's question based strictly and only on the provided course book context chunks below.

Guidelines for answering:
1. Ground every statement strictly in the provided context. If the context does not contain the answer, say clearly: "I cannot find the answer in the provided B.Tech Course-book." Do not make up facts.
2. For EVERY claim or piece of information you write, you MUST include a clean inline citation citing the exact page number of the source chunk in the format '[Page X]'. 
   Example: 'Students must complete 160 credits to graduate [Page 12].'
3. Maintain a highly professional, academic, and structured tone. Use markdown bullet points and tables where appropriate to present credit maps, syllabi, or rules clearly.
4. When listing courses, always include course codes (e.g., CS-302) if they are present in the text.
"""

    prompt = f"""{system_prompt}

Retrieved Context Chunks:
=======================================
{context_text}
=======================================

Student Query: {query}

Synthesized Answer (remember to cite exact Page numbers inline):"""

    response = llm.invoke(prompt)
    return response.content

def query_assistant(query):
    print(f"💬 User: {query}\n")
    expanded = expand_query(query)
    print(f"🔄 Expanded queries: {expanded[1:]}")
    retrieved = perform_hybrid_search(expanded, top_n=4)
    print(f"📚 Retrieved {len(retrieved)} parent chunks.")
    
    print("🤖 Generating answer...")
    answer = generate_answer(query, retrieved)
    
    print("\n" + "="*70)
    print("✨ ADVANCED RAG RESPONSE:")
    print("="*70)
    print(answer)
    print("="*70)
    
    pages = sorted(list(dict.fromkeys([doc.metadata["page"] for doc in retrieved])))
    print(f"📖 Source Pages Referenced: {', '.join([f'Page {p}' for p in pages])}\n")

## 🧪 Step 6: Test Queries
Let's test our Advanced RAG system with a query!

In [ ]:
query_assistant("What is the structure of the Summer Semester?")

In [ ]:
query_assistant("What are the requirements or details of core courses in the third semester?")

## 💬 Chat Room
Run the cell below to chat live with your B.Tech Course-book RAG Assistant!

In [ ]:
print("🎓 Chat Assistant ready! Type 'exit' to quit.")
print("-"*50)
while True:
    q = input("Student: ")
    if q.lower() == 'exit':
        print("Goodbye!")
        break
    if not q.strip():
        continue
    query_assistant(q)